In [1]:
!pip install pulp

import pulp as pl


model = pl.LpProblem("Production_Inventory_Optimization", pl.LpMinimize)

months = ["January", "February", "March", "April"]
products = {
    "A": {"time": 1.2, "cost": 20, "holding": 1},
    "B": {"time": 1.3, "cost": 30, "holding": 2},
    "C": {"time": 1.0, "cost": 10, "holding": 0.5},
}

hours_available = [500, 450, 400, 350]
demand = {
    "January": {"A": 100, "B": 80, "C": 120},
    "February": {"A": 110, "B": 90, "C": 130},
    "March": {"A": 120, "B": 100, "C": 140},
    "April": {"A": 130, "B": 110, "C": 150},
}


X = pl.LpVariable.dicts(
    "Production", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)
I = pl.LpVariable.dicts(
    "Inventory", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)


model += pl.lpSum(
    [products[p]["cost"] * X[(p, t)] + products[p]["holding"] * I[(p, t)]
     for p in products for t in months]
)


for p in products:
    for idx, t in enumerate(months):
        prev_inventory = 0 if idx == 0 else I[(p, months[idx - 1])]
        model += I[(p, t)] == prev_inventory + X[(p, t)] - demand[t][p]


for idx, t in enumerate(months):
    model += pl.lpSum([products[p]["time"] * X[(p, t)] for p in products]) <= hours_available[idx]


model.solve()


status = pl.LpStatus[model.status]
total_cost = pl.value(model.objective)
production_plan = {p: {t: X[(p, t)].varValue for t in months} for p in products}
inventory_plan = {p: {t: I[(p, t)].varValue for t in months} for p in products}

status, total_cost, production_plan, inventory_plan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 37.5 MB/s eta 0:00:00


('Optimal',
 26127.0,
 {'A': {'January': 100.0, 'February': 110.0, 'March': 120.0, 'April': 130.0},
  'B': {'January': 80.0, 'February': 90.0, 'March': 100.0, 'April': 110.0},
  'C': {'January': 162.0, 'February': 201.0, 'March': 126.0, 'April': 51.0}},
 {'A': {'January': 0.0, 'February': 0.0, 'March': 0.0, 'April': 0.0},
  'B': {'January': 0.0, 'February': 0.0, 'March': 0.0, 'April': 0.0},
  'C': {'January': 42.0, 'February': 113.0, 'March': 99.0, 'April': 0.0}})

**Part B**

In [2]:

!pip install pulp


import pulp as pl


model_b = pl.LpProblem("Production_Inventory_Optimization_Part_B", pl.LpMinimize)

months = ["January", "February", "March", "April"]
products = {
    "A": {"time": 1.2, "cost": 20, "holding": 1},
    "B": {"time": 1.3, "cost": 30, "holding": 2},
    "C": {"time": 1.0, "cost": 10, "holding": 0.5},
}

hours_available = [500, 450, 400, 350]
demand = {
    "January": {"A": 100, "B": 80, "C": 120},
    "February": {"A": 110, "B": 90, "C": 130},
    "March": {"A": 120, "B": 100, "C": 140},
    "April": {"A": 130, "B": 110, "C": 150},
}


adjusted_costs = {
    (p, t): products[p]["cost"] + (5 if p == "B" and t == "March" else 0)
    for p in products for t in months
}


X = pl.LpVariable.dicts(
    "Production", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)
I = pl.LpVariable.dicts(
    "Inventory", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)


model_b += pl.lpSum(
    [adjusted_costs[(p, t)] * X[(p, t)] + products[p]["holding"] * I[(p, t)]
     for p in products for t in months]
)


for p in products:
    for idx, t in enumerate(months):
        prev_inventory = 0 if idx == 0 else I[(p, months[idx - 1])]
        model_b += I[(p, t)] == prev_inventory + X[(p, t)] - demand[t][p]


for idx, t in enumerate(months):
    model_b += pl.lpSum([products[p]["time"] * X[(p, t)] for p in products]) <= hours_available[idx]


model_b.solve()


status_b = pl.LpStatus[model_b.status]
total_cost_b = pl.value(model_b.objective)
production_plan_b = {p: {t: X[(p, t)].varValue for t in months} for p in products}
inventory_plan_b = {p: {t: I[(p, t)].varValue for t in months} for p in products}

print("Status:", status_b)
print("Total Cost:", total_cost_b)
print("\nProduction Plan:")
for p in products:
    for t in months:
        print(f"  Production of {p} in {t}: {production_plan_b[p][t]}")
        print(f"  Inventory of {p} in {t}: {inventory_plan_b[p][t]}")

Status: Optimal
Total Cost: 26279.0

Production Plan:
  Production of A in January: 100.0
  Inventory of A in January: 0.0
  Production of A in February: 110.0
  Inventory of A in February: 0.0
  Production of A in March: 120.0
  Inventory of A in March: 0.0
  Production of A in April: 130.0
  Inventory of A in April: 0.0
  Production of B in January: 80.0
  Inventory of B in January: 0.0
  Production of B in February: 190.0
  Inventory of B in February: 100.0
  Production of B in March: 0.0
  Inventory of B in March: 0.0
  Production of B in April: 110.0
  Inventory of B in April: 0.0
  Production of C in January: 179.0
  Inventory of C in January: 59.0
  Production of C in February: 71.0
  Inventory of C in February: 0.0
  Production of C in March: 239.0
  Inventory of C in March: 99.0
  Production of C in April: 51.0
  Inventory of C in April: 0.0


**Part- C**

In [3]:

!pip install pulp
import pulp as pl


model_c = pl.LpProblem("Production_Inventory_Optimization_Part_C", pl.LpMinimize)
months = ["January", "February", "March", "April"]
products = {
    "A": {"time": 1.2, "cost": 20, "holding": 1},
    "B": {"time": 1.3, "cost": 30, "holding": 2},
    "C": {"time": 1.0, "cost": 10, "holding": 0.5},
}

hours_available = [500, 450, 400, 350]
demand = {
    "January": {"A": 100, "B": 80, "C": 120},
    "February": {"A": 110, "B": 90, "C": 130},
    "March": {"A": 120, "B": 100, "C": 140},
    "April": {"A": 130, "B": 110, "C": 170},
}


X = pl.LpVariable.dicts(
    "Production", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)
I = pl.LpVariable.dicts(
    "Inventory", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)


model_c += pl.lpSum(
    [products[p]["cost"] * X[(p, t)] + products[p]["holding"] * I[(p, t)]
     for p in products for t in months]
)


for p in products:
    for idx, t in enumerate(months):
        prev_inventory = 0 if idx == 0 else I[(p, months[idx - 1])]
        model_c += I[(p, t)] == prev_inventory + X[(p, t)] - demand[t][p]


for idx, t in enumerate(months):
    model_c += pl.lpSum([products[p]["time"] * X[(p, t)] for p in products]) <= hours_available[idx]


model_c.solve()


status_c = pl.LpStatus[model_c.status]
total_cost_c = pl.value(model_c.objective)
production_plan_c = {p: {t: X[(p, t)].varValue for t in months} for p in products}
inventory_plan_c = {p: {t: I[(p, t)].varValue for t in months} for p in products}

print("Status:", status_c)
print("Total Cost:", total_cost_c)
print("\nProduction Plan:")
for p in products:
    for t in months:
        print(f"  Production of {p} in {t}: {production_plan_c[p][t]}")
        print(f"  Inventory of {p} in {t}: {inventory_plan_c[p][t]}")

Status: Optimal
Total Cost: 26357.0

Production Plan:
  Production of A in January: 100.0
  Inventory of A in January: 0.0
  Production of A in February: 110.0
  Inventory of A in February: 0.0
  Production of A in March: 120.0
  Inventory of A in March: 0.0
  Production of A in April: 130.0
  Inventory of A in April: 0.0
  Production of B in January: 80.0
  Inventory of B in January: 0.0
  Production of B in February: 90.0
  Inventory of B in February: 0.0
  Production of B in March: 100.0
  Inventory of B in March: 0.0
  Production of B in April: 110.0
  Inventory of B in April: 0.0
  Production of C in January: 182.0
  Inventory of C in January: 62.0
  Production of C in February: 201.0
  Inventory of C in February: 133.0
  Production of C in March: 126.0
  Inventory of C in March: 119.0
  Production of C in April: 51.0
  Inventory of C in April: 0.0


**Part - D**

In [4]:
!pip install pulp
import pulp as pl

model_d = pl.LpProblem("Production_Inventory_Optimization_Part_D", pl.LpMinimize)
months = ["January", "February", "March", "April"]
products = {
    "A": {"time": 1.2, "cost": 20, "holding": 1},
    "B": {"time": 1.3, "cost": 30, "holding": 2},
    "C": {"time": 1.0, "cost": 10, "holding": 0.5},
}

hours_available = [500, 450, 400, 350]
demand = {
    "January": {"A": 100, "B": 80, "C": 120},
    "February": {"A": 110, "B": 90, "C": 130},
    "March": {"A": 120, "B": 100, "C": 140},
    "April": {"A": 130, "B": 110, "C": 150},
}

X = pl.LpVariable.dicts(
    "Production", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)
I = pl.LpVariable.dicts(
    "Inventory", ((p, t) for p in products for t in months), lowBound=0, cat="Integer"
)


model_d += pl.lpSum(
    [products[p]["cost"] * X[(p, t)] + products[p]["holding"] * I[(p, t)]
     for p in products for t in months]
)


for p in products:
    for idx, t in enumerate(months):
        prev_inventory = 0 if idx == 0 else I[(p, months[idx - 1])]
        model_d += I[(p, t)] == prev_inventory + X[(p, t)] - demand[t][p]


for idx, t in enumerate(months):
    model_d += pl.lpSum([products[p]["time"] * X[(p, t)] for p in products]) <= hours_available[idx]


model_d += pl.lpSum([products[p]["holding"] * I[(p, t)] for p in products for t in months]) <= 500, "InventoryCostLimit"


model_d.solve()


status_d = pl.LpStatus[model_d.status]
total_cost_d = pl.value(model_d.objective)
production_plan_d = {p: {t: X[(p, t)].varValue for t in months} for p in products}
inventory_plan_d = {p: {t: I[(p, t)].varValue for t in months} for p in products}

print("Status:", status_d)
print("Total Cost:", total_cost_d)
print("\nProduction Plan:")
for p in products:
    for t in months:
        print(f"  Production of {p} in {t}: {production_plan_d[p][t]}")
        print(f"  Inventory of {p} in {t}: {inventory_plan_d[p][t]}")

Status: Optimal
Total Cost: 26127.0

Production Plan:
  Production of A in January: 100.0
  Inventory of A in January: 0.0
  Production of A in February: 110.0
  Inventory of A in February: 0.0
  Production of A in March: 120.0
  Inventory of A in March: 0.0
  Production of A in April: 130.0
  Inventory of A in April: 0.0
  Production of B in January: 80.0
  Inventory of B in January: 0.0
  Production of B in February: 90.0
  Inventory of B in February: 0.0
  Production of B in March: 100.0
  Inventory of B in March: 0.0
  Production of B in April: 110.0
  Inventory of B in April: 0.0
  Production of C in January: 162.0
  Inventory of C in January: 42.0
  Production of C in February: 201.0
  Inventory of C in February: 113.0
  Production of C in March: 126.0
  Inventory of C in March: 99.0
  Production of C in April: 51.0
  Inventory of C in April: 0.0
